# A1-1 쇼핑몰 단골 찾기 — 분석 리포트

H&M Personalized Fashion Recommendations(Kaggle) 데이터로 멀티모달 EDA 와 RFM 고객 세분화를 한다.
분석 로직은 `src/pipeline.py` 의 `DataAnalyzer` 클래스에 있고, 이 노트북은 그 결과를 읽고 해석한다.

In [1]:
import sys
from pathlib import Path

# 노트북은 notebooks/ 에서 실행되므로 프로젝트 루트를 import 경로에 넣는다
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd
from src.pipeline import DataAnalyzer

pd.set_option("display.max_columns", 30)

## 1. 데이터 로드 및 기본 탐색

### 표본 기준

| 항목 | 기준 |
|---|---|
| 기간 | 2019-09-23 ~ 2020-09-22 (최근 12개월) |
| 고객 | 기간 안에 1번 이상 산 고객 중 무작위 1만 명 (`random_state=42`) |
| 거래 | 뽑힌 고객의 기간 내 거래 줄 **전부** — 줄을 뽑으면 고객마다 F·M 이 줄어든다 |
| 결합 | 거래 줄마다 상품(`articles`)·고객(`customers`) 정보를 왼쪽 결합 |
| Recency 기준일 | 2020-09-22 (데이터 마지막 거래일) |

In [2]:
analyzer = DataAnalyzer()
df = analyzer.load_data()
analyzer.load_summary

{'start_date': '2019-09-23',
 'end_date': '2020-09-22',
 'random_state': 42,
 'rows_read': 31788324,
 'window_rows': 14898423,
 'window_customers': 994320,
 'sampled_customers': 10000,
 'rows': 149548,
 'articles_unmatched': 0,
 'customers_unmatched': 0,
 'seconds': 8.7}

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 149548 entries, 0 to 149547
Data columns (total 23 columns):
 #   Column                        Non-Null Count   Dtype         
---  ------                        --------------   -----         
 0   t_dat                         149548 non-null  datetime64[us]
 1   customer_id                   149548 non-null  str           
 2   article_id                    149548 non-null  int64         
 3   price                         149548 non-null  float64       
 4   sales_channel_id              149548 non-null  int8          
 5   prod_name                     149548 non-null  str           
 6   product_type_name             149548 non-null  str           
 7   product_group_name            149548 non-null  str           
 8   graphical_appearance_name     149548 non-null  str           
 9   colour_group_name             149548 non-null  str           
 10  perceived_colour_value_name   149548 non-null  str           
 11  perceived_colour_master_

**해석 — 구조와 결측**

- 149,548줄 × 23열로 과제 기준(1,000건·8열)을 넘는다. 날짜형(`t_dat`)·수치형(`price`·`age`)·범주형(`product_type_name` 등 11열)·텍스트(`prod_name`·`detail_desc`)가 있고, 이미지는 사진 확보 후 붙인다.
- 결측은 고객 정보에 몰려 있다: `FN` 83,493줄(55.8%)·`Active` 84,122줄(56.3%), `fashion_news_frequency` 671줄, `age` 431줄(0.3%), `club_member_status` 279줄, `detail_desc` 157줄.
- **시사점**: 절반 넘게 빈 열은 대치하기 전에 빈칸이 무슨 뜻인지부터 확인해야 한다.

In [4]:
df.describe()

,t_dat,article_id,price,sales_channel_id,FN,Active,age
count,149548,1.495480e+05,149548.000000,149548.000000,66055.0,65426.0,149117.000000
mean,2020-03-29 11:49:18.129831,7.526006e+08,0.028009,1.714125,1.0,1.0,35.532468
min,2019-09-23 00:00:00,1.087750e+08,0.000424,1.000000,1.0,1.0,17.000000
25%,2019-12-24 00:00:00,7.060160e+08,0.016542,1.000000,1.0,1.0,25.000000
50%,2020-04-09 00:00:00,7.872255e+08,0.025407,2.000000,1.0,1.0,31.000000
75%,2020-06-26 00:00:00,8.362840e+08,0.033881,2.000000,1.0,1.0,47.000000
max,2020-09-22 00:00:00,9.534500e+08,0.422034,2.000000,1.0,1.0,99.000000
std,NaN,1.302270e+08,0.019107,0.451832,0.0,0.0,13.358036


**해석 — 수치형 분포**

- `price` 는 평균 0.0280 > 중앙값 0.0254 이고, 최댓값 0.4220 이 Q3(0.0339)의 약 12배다 → 오른쪽 꼬리가 긴 분포.
- `age` 는 중앙값 31세, Q1 25 ~ Q3 47. 단 이 표는 **거래 줄 기준**이라 자주 사는 고객의 나이가 여러 번 세어진다.
- `FN`·`Active` 는 값이 있는 줄이 전부 1.0(std 0)이다.
- **시사점**: 금액도 소수에 쏠릴 수 있어 이상치 처리 전후 비교가 필요하고, 고객 특성은 고객 단위로 묶어 다시 봐야 한다.

In [5]:
df.describe(include="str")

,customer_id,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name,detail_desc,club_member_status,fashion_news_frequency
count,149548,149548,149548,149548,149548,149548,149548,149548,149548,149548,149548,149548,149548,149391,149269,148877
unique,10000,14685,110,15,29,50,8,19,230,10,5,55,21,13737,3,2
top,92838b6470bf09ea2247a7f244a059e13eec4224cfe93d...,Jade HW Skinny Denim TRS,Trousers,Garment Upper body,Solid,Black,Dark,Black,Swimwear,Ladieswear,Ladieswear,Womens Everyday Collection,Jersey Fancy,High-waisted jeans in washed superstretch deni...,ACTIVE,NONE
freq,462,1089,19771,57944,88522,51990,70672,51874,10747,62482,98193,28172,23342,1155,147015,82504


**해석 — 범주형·식별자**

- `customer_id` 고유값 10,000 = 뽑은 고객 수. 고객당 평균 15.0줄인데 최다 고객은 462줄로 약 31배다.
- `index_group_name` 5개 중 Ladieswear 가 98,193줄(65.7%), `product_type_name` 110종 중 Trousers 가 19,771줄(13.2%)로 가장 많다.
- **시사점**: RFM 에서 극단적인 고객을 어떻게 다룰지 정해야 하고, 세그먼트 전략은 여성복 중심으로 해석된다.

In [6]:
df.head()

,t_dat,customer_id,article_id,price,sales_channel_id,prod_name,product_type_name,product_group_name,graphical_appearance_name,colour_group_name,perceived_colour_value_name,perceived_colour_master_name,department_name,index_name,index_group_name,section_name,garment_group_name,detail_desc,FN,Active,club_member_status,fashion_news_frequency,age
0,2019-09-23,04e2301dc0581c5b443429e1614cf21ffe9b23d8d3fe05...,740943002,0.013542,2,Pamela,Dress,Garment Full body,Stripe,Off White,Dusty Light,White,Dress,Ladieswear,Ladieswear,Womens Everyday Collection,Dresses Ladies,Short dress in a viscose weave with a gentle V...,1.0,1.0,ACTIVE,Regularly,34.0
1,2019-09-23,04e2301dc0581c5b443429e1614cf21ffe9b23d8d3fe05...,660497001,0.018627,2,Emery(1),Dress,Garment Full body,Solid,Black,Dark,Black,Jersey fancy,Ladieswear,Ladieswear,Womens Everyday Collection,Jersey Fancy,Short sleeveless dress in soft slub jersey mad...,1.0,1.0,ACTIVE,Regularly,34.0
2,2019-09-23,04e2301dc0581c5b443429e1614cf21ffe9b23d8d3fe05...,727611003,0.025407,2,Zidane jumpsuit,Jumpsuit/Playsuit,Garment Full body,All over pattern,Yellow,Medium,Yellow,Dresses,Divided,Divided,Divided Collection,Dresses Ladies,V-neck playsuit in a crinkled weave with a wra...,1.0,1.0,ACTIVE,Regularly,34.0
3,2019-09-23,04e2301dc0581c5b443429e1614cf21ffe9b23d8d3fe05...,727611003,0.025407,2,Zidane jumpsuit,Jumpsuit/Playsuit,Garment Full body,All over pattern,Yellow,Medium,Yellow,Dresses,Divided,Divided,Divided Collection,Dresses Ladies,V-neck playsuit in a crinkled weave with a wra...,1.0,1.0,ACTIVE,Regularly,34.0
4,2019-09-23,04e2301dc0581c5b443429e1614cf21ffe9b23d8d3fe05...,740943001,0.013542,2,Pamela,Dress,Garment Full body,All over pattern,Dark Orange,Dark,Beige,Dress,Ladieswear,Ladieswear,Womens Everyday Collection,Dresses Ladies,Short dress in a viscose weave with a gentle V...,1.0,1.0,ACTIVE,Regularly,34.0


**해석 — 실제 줄 모양**

- 인덱스 2·3 은 같은 고객·같은 날·같은 상품(`727611003`)·같은 가격이다. Kaggle 데이터 설명대로 **같은 상품을 2개 산 것**을 뜻하는 중복 줄이다(이 데이터에는 수량 열이 없다).
- 다섯 줄 모두 같은 고객·같은 날이라, 주문 정의(고객 + 날짜)로는 주문 1건이다.
- **시사점**: 중복 줄을 지우면 M(금액)이 줄어들므로 지우지 않는다.